# 01 Stationarity and Differencing

By the end of this notebook, you should be able to:

- describe the five-step Box-Jenkins workflow;
- explain weak stationarity in operational terms;
- compute first and second differences;
- choose a working series for a nonseasonal ARIMA analysis.

In [1]:
from lite_setup import ensure_packages
await ensure_packages()

Running outside JupyterLite; assuming packages are already installed.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check, check_between, check_close, check_columns
from boxjenkins_utils import (
    first_difference, second_difference, seasonal_difference, regular_then_seasonal_difference,
    acf_pacf_table, mean_zero_t_test, fit_arima, fit_sarima, parameter_table,
    forecast_table, ljung_box_table, arima_grid_search, plot_series,
    plot_acf_pacf_pair, plot_forecast,
)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_process import ArmaProcess
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')
towel = pd.read_csv(DATA_DIR / 'paper_towel_sales.csv')
y = towel['Towel_Sales']
towel.head()

,Week,Towel_Sales
0,1,15.0000
1,2,14.4064
2,3,14.9383
3,4,16.0374
4,5,15.6320


## The Box-Jenkins Workflow

Box-Jenkins models are built for stationary time series. In practice the workflow is:

1. Transform the observed series if needed so the working series is approximately stationary.
2. Identify a tentative ARIMA model from ACF/PACF behavior.
3. Estimate the model parameters.
4. Check whether residuals look like white noise; if not, revise the model.
5. Once the model is adequate, forecast future values and communicate uncertainty.

This is iterative. A model selected from ACF/PACF is tentative until diagnostics support it.

## Weak Stationarity

A time series is weakly stationary when its mean is constant over time and its autocovariance depends only on the lag, not on calendar time:

$$E(y_t)=\mu,$$

$$\operatorname{Cov}(y_t,y_{t+k})=\gamma(k).$$

This implies a constant variance because variance is the autocovariance at lag 0. A trend, changing level, changing variance, or regular seasonal pattern is a warning sign that the raw series may not be stationary.

In [3]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=False)
axes[0].plot(towel['Week'], y, marker='o')
axes[0].set_title('Paper towel sales')
axes[0].set_ylabel('Sales')
z1 = first_difference(y)
axes[1].plot(np.arange(2, len(y) + 1), z1, marker='o', color='tab:orange')
axes[1].set_title('First differences: $z_t = y_t - y_{t-1}$')
axes[1].set_ylabel('Difference')
z2 = second_difference(y)
axes[2].plot(np.arange(3, len(y) + 1), z2, marker='o', color='tab:green')
axes[2].set_title('Second differences: $(y_t-y_{t-1})-(y_{t-1}-y_{t-2})$')
axes[2].set_xlabel('Week')
axes[2].set_ylabel('Difference')
plt.tight_layout()

## Differencing Formulas

The first difference is

$$z_t = y_t-y_{t-1}, \quad t=2,\ldots,n.$$

If first differences are still nonstationary, a second difference can be used:

$$z_t = (y_t-y_{t-1})-(y_{t-1}-y_{t-2}) = y_t - 2y_{t-1} + y_{t-2}, \quad t=3,\ldots,n.$$

The transformed sequence is called the working series. Its first index may be 2 or 3 because differencing consumes observations.

In [4]:
manual_first = y.iloc[1:].to_numpy() - y.iloc[:-1].to_numpy()
manual_second = y.iloc[2:].to_numpy() - 2*y.iloc[1:-1].to_numpy() + y.iloc[:-2].to_numpy()
print(check(np.allclose(manual_first, z1.to_numpy()), 'PASS: first differences match the formula.'))
print(check(np.allclose(manual_second, z2.to_numpy()), 'PASS: second differences match the formula.'))
print(f'Original length: {len(y)}')
print(f'First-difference length: {len(z1)}')
print(f'Second-difference length: {len(z2)}')

PASS: first differences match the formula.
PASS: second differences match the formula.
Original length: 120
First-difference length: 119
Second-difference length: 118


## How To Decide Whether Differencing Helped

Use plots first, then ACF/PACF in the next notebook.

- If the raw series wanders for long stretches, trends, or has an ACF that dies down extremely slowly, it is likely nonstationary.
- If a differenced series fluctuates around a stable mean with roughly constant spread and short memory, it is a better working series.
- Do not difference mechanically. Extra differencing can create unnecessary negative autocorrelation and make forecasts worse.

For the towel data, the raw series has long runs above and below its overall level. The first difference is a more plausible working series for nonseasonal Box-Jenkins modeling.

## Practice Prompt

In one or two sentences, explain what information is lost when we difference a series and what information becomes easier to model.

Reference idea: differencing removes the original level scale and focuses the model on changes. That can make the working series stationary, but forecasts must be interpreted back on the original series if the business question is about future levels.